<a href="https://colab.research.google.com/github/leonbenenati/Projeto-Avaliativo---M-dulo-2/blob/feature%2Fteste-api/Projeto_Avaliativo_M%C3%B3dulo_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from lightgbm import LGBMClassifier

In [34]:
df = pd.read_csv("/content/manutencao_preditiva.csv")

In [35]:
df

,udi,id_produto,tipo,temperatura_ar_k,temperatura_processo_k,velocidade_rotacao_rpm,torque_nm,desgaste_ferramenta_min,falha_maquina,falha_twf,falha_hdf,falha_pwf,falha_osf,falha_rnf
0,1,M14860,M,298.1,308.6,1551.0,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408.0,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498.0,49.4,5,0,0,0,0,0,0
3,4,L47183,L,NaN,NaN,NaN,NaN,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408.0,40.0,9,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,M24855,M,298.8,308.4,1604.0,29.5,14,0,0,0,0,0,0
9996,9997,H39410,H,298.9,308.4,1632.0,31.8,17,0,0,0,0,0,0
9997,9998,M24857,M,299.0,308.6,1645.0,33.4,22,0,0,0,0,0,0
9998,9999,H39412,H,299.0,308.7,1408.0,48.5,25,0,0,0,0,0,0


In [46]:
def criar_features(df):


    df["potencia"] = (
        df["velocidade_rotacao_rpm"] *
        df["torque_nm"]
    )

    df["delta_temperatura"] = (
        df["temperatura_processo_k"] -
        df["temperatura_ar_k"]
    )

    df["desgaste_torque"] = (
        df["desgaste_ferramenta_min"] *
        df["torque_nm"]
    )

    return df

In [47]:
criar_features(df)

,udi,id_produto,tipo,temperatura_ar_k,temperatura_processo_k,velocidade_rotacao_rpm,torque_nm,desgaste_ferramenta_min,falha_maquina,falha_twf,falha_hdf,falha_pwf,falha_osf,falha_rnf,potencia,delta_temperatura,desgaste_torque
0,1,M14860,M,298.1,308.6,1551.0,42.8,0,0,0,0,0,0,0,66382.8,10.5,0.0
1,2,L47181,L,298.2,308.7,1408.0,46.3,3,0,0,0,0,0,0,65190.4,10.5,138.9
2,3,L47182,L,298.1,308.5,1498.0,49.4,5,0,0,0,0,0,0,74001.2,10.4,247.0
3,4,L47183,L,NaN,NaN,NaN,NaN,7,0,0,0,0,0,0,NaN,NaN,NaN
4,5,L47184,L,298.2,308.7,1408.0,40.0,9,0,0,0,0,0,0,56320.0,10.5,360.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,M24855,M,298.8,308.4,1604.0,29.5,14,0,0,0,0,0,0,47318.0,9.6,413.0
9996,9997,H39410,H,298.9,308.4,1632.0,31.8,17,0,0,0,0,0,0,51897.6,9.5,540.6
9997,9998,M24857,M,299.0,308.6,1645.0,33.4,22,0,0,0,0,0,0,54943.0,9.6,734.8
9998,9999,H39412,H,299.0,308.7,1408.0,48.5,25,0,0,0,0,0,0,68288.0,9.7,1212.5


In [49]:
colunas = [
    "tipo",
    "temperatura_ar_k",
    "temperatura_processo_k",
    "velocidade_rotacao_rpm",
    "torque_nm",
    "desgaste_ferramenta_min",
    "potencia",
    "delta_temperatura",
    "desgaste_torque",
]

X = df[colunas].copy()
y = df["falha_maquina"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42)

In [50]:
df["desgaste_ferramenta_min"].isna().sum()

np.int64(0)

In [58]:
print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (7000, 9)
Teste: (3000, 9)


In [52]:


# Variáveis que receberão imputação pela média
media_cols = [
    "temperatura_ar_k",
    "temperatura_processo_k",
    "torque_nm"
]

# Variável que receberá imputação pela mediana
mediana_cols = [
    "velocidade_rotacao_rpm"
]

# Imputação da média
imp_media = SimpleImputer(strategy="mean")

# Imputação da mediana
imp_mediana = SimpleImputer(strategy="median")

# Codificação da variável categórica
codificador = OneHotEncoder(
    handle_unknown="ignore",
    drop="first",
    sparse_output=False
)

# Organizando todas as transformações
preprocessamento = ColumnTransformer(
    transformers=[
        ("media", imp_media, media_cols),
        ("mediana", imp_mediana, mediana_cols),
        ("tipo", codificador, ["tipo"])
    ],
    remainder="passthrough"
)

In [56]:
X_train_pre = preprocessamento.fit_transform(X_train)

X_test_pre = preprocessamento.transform(X_test)

In [60]:
print("Treino:", X_train_pre.shape)
print("Teste:", X_test_pre.shape)

Treino: (7000, 10)
Teste: (3000, 10)


In [61]:
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

modelo = LGBMClassifier(
    random_state=42,
    verbosity=-1
)

modelo.fit(X_train_pre, y_train)

y_pred = modelo.predict(X_test_pre)
y_prob = modelo.predict_proba(X_test_pre)[:, 1]

print("Acurácia:", accuracy_score(y_test, y_pred))
print("Precisão:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Acurácia: 0.9893333333333333
Precisão: 0.8977272727272727
Recall: 0.7745098039215687
F1-Score: 0.8315789473684211
ROC-AUC: 0.9791607464241736


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [62]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

pipeline = Pipeline([
    ("preprocessamento", preprocessamento),
    ("modelo", LGBMClassifier(
        random_state=42,
        verbosity=-1
    ))
])

In [66]:


parametros = {
    "modelo__n_estimators": [100, 200, 300],
    "modelo__learning_rate": [0.03, 0.05, 0.1],
    "modelo__num_leaves": [15, 31, 50],
    "modelo__max_depth": [-1, 5, 10],
    "modelo__min_child_samples": [20, 50, 100],
    "modelo__subsample": [0.8, 1.0],
    "modelo__colsample_bytree": [0.8, 1.0]
}

cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

busca = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=parametros,
    n_iter=10,
    scoring="f1",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

busca.fit(X_train, y_train)

Fitting 10 folds for each of 10 candidates, totalling 100 fits


/usr/local/lib/python3.13/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


RandomizedSearchCV(cv=StratifiedKFold(n_splits=10, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('preprocessamento',
                                              ColumnTransformer(remainder='passthrough',
                                                                transformers=[('media',
                                                                               SimpleImputer(),
                                                                               ['temperatura_ar_k',
                                                                                'temperatura_processo_k',
                                                                                'torque_nm']),
                                                                              ('mediana',
                                                                               SimpleImputer(strategy='median'),
                                                                               ['velocidade_rotacao_rpm']),
                                                                              ('tipo',
                                                                               OneH...
                                              LGBMClassifier(random_state=42,
                                                             verbosity=-1))]),
                   n_jobs=-1,
                   param_distributions={'modelo__colsample_bytree': [0.8, 1.0],
                                        'modelo__learning_rate': [0.03, 0.05,
                                                                  0.1],
                                        'modelo__max_depth': [-1, 5, 10],
                                        'modelo__min_child_samples': [20, 50,
                                                                      100],
                                        'modelo__n_estimators': [100, 200, 300],
                                        'modelo__num_leaves': [15, 31, 50],
                                        'modelo__subsample': [0.8, 1.0]},
                   random_state=42, scoring='f1', verbose=1)

In [67]:
print("Melhor F1-Score:", busca.best_score_)
print("Melhores parâmetros:", busca.best_params_)

Melhor F1-Score: 0.8499711479096913
Melhores parâmetros: {'modelo__subsample': 1.0, 'modelo__num_leaves': 15, 'modelo__n_estimators': 200, 'modelo__min_child_samples': 20, 'modelo__max_depth': 10, 'modelo__learning_rate': 0.05, 'modelo__colsample_bytree': 0.8}


In [68]:
import joblib

modelo_final = busca.best_estimator_

y_pred = modelo_final.predict(X_test)
y_prob = modelo_final.predict_proba(X_test)[:, 1]

print("Acurácia:", accuracy_score(y_test, y_pred))
print("Precisão:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

joblib.dump(modelo_final, "modelo_falhas.joblib")

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Acurácia: 0.9893333333333333
Precisão: 0.9069767441860465
Recall: 0.7647058823529411
F1-Score: 0.8297872340425532
ROC-AUC: 0.9752127904301817


['modelo_falhas.joblib']

# API

## schemas.py

In [89]:
%%writefile schemas.py

from pydantic import BaseModel, Field, ConfigDict
from typing import Literal


class DadosEntrada(BaseModel):
    model_config = ConfigDict(extra="forbid")

    tipo: Literal["L", "M", "H"] = Field(
    ...,
    description="Tipo da máquina: L, M ou H"
)

    temperatura_ar_k: float = Field(
        ...,
        description="Temperatura do ar em Kelvin"
    )

    temperatura_processo_k: float = Field(
        ...,
        description="Temperatura do processo em Kelvin"
    )

    velocidade_rotacao_rpm: float = Field(
        ...,
        ge=0,
        description="Velocidade de rotação em RPM"
    )

    torque_nm: float = Field(
        ...,
        ge=0,
        description="Torque em Newton-metro"
    )

    desgaste_ferramenta_min: float = Field(
        ...,
        ge=0,
        description="Desgaste da ferramenta em minutos"
    )


class ResultadoPredicao(BaseModel):
    falha_maquina: int = Field(
        ...,
        ge=0,
        le=1,
        description="Predição: 0 para ausência de falha e 1 para falha"
    )

    probabilidade_falha: float = Field(
        ...,
        ge=0,
        le=1,
        description="Probabilidade estimada de falha"
    )

Overwriting schemas.py


## services.py

In [74]:
%%writefile services.py

import pandas as pd


def criar_features(dados: dict) -> pd.DataFrame:
    df = pd.DataFrame([dados])

    df["potencia"] = (
        df["velocidade_rotacao_rpm"] *
        df["torque_nm"]
    )

    df["delta_temperatura"] = (
        df["temperatura_processo_k"] -
        df["temperatura_ar_k"]
    )

    df["desgaste_torque"] = (
        df["desgaste_ferramenta_min"] *
        df["torque_nm"]
    )

    return df


def realizar_predicao(modelo, dados: dict) -> dict:
    entrada = criar_features(dados)

    predicao = int(modelo.predict(entrada)[0])

    probabilidade = float(
        modelo.predict_proba(entrada)[0, 1]
    )

    return {
        "falha_maquina": predicao,
        "probabilidade_falha": probabilidade
    }

Overwriting services.py


## routers.py

In [75]:
%%writefile routers.py

from fastapi import APIRouter, HTTPException, Request

from schemas import DadosEntrada, ResultadoPredicao
from services import realizar_predicao

router = APIRouter()


@router.get("/health")
def health(request: Request):
    modelo = getattr(request.app.state, "modelo", None)

    return {
        "status": "ok",
        "modelo_carregado": modelo is not None
    }


@router.post(
    "/predict",
    response_model=ResultadoPredicao
)
def predict(dados: DadosEntrada, request: Request):
    modelo = getattr(request.app.state, "modelo", None)

    if modelo is None:
        raise HTTPException(
            status_code=500,
            detail="Modelo não carregado"
        )

    return realizar_predicao(
        modelo,
        dados.model_dump()
    )

Writing routers.py


## main.py

In [76]:
%%writefile main.py

from contextlib import asynccontextmanager
from pathlib import Path

import joblib
from fastapi import FastAPI

from routers import router


CAMINHO_MODELO = Path(__file__).parent / "modelo_falhas.joblib"


@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.modelo = joblib.load(CAMINHO_MODELO)

    yield

    app.state.modelo = None


app = FastAPI(
    title="API de Predição de Falhas em Máquinas",
    description="API para identificar falhas em máquinas industriais usando LightGBM.",
    version="1.0.0",
    lifespan=lifespan
)

app.include_router(router)

Writing main.py


# Teste API

In [93]:
%pip install fastapi uvicorn nest-asyncio

#Essas bibliotecas permitem executar a API no ambiente do Colab. O nest-asyncio será utilizado para iniciar o servidor dentro do notebook.

In [94]:
import nest_asyncio
import threading
import uvicorn

nest_asyncio.apply()

config = uvicorn.Config(
    "main:app",
    host="127.0.0.1",
    port=8000,
    log_level="info"
)

servidor = uvicorn.Server(config)

thread = threading.Thread(
    target=servidor.run,
    daemon=True
)

thread.start()

INFO:     Started server process [1439]
INFO:     Waiting for application startup.


In [95]:
import requests

resposta = requests.get("http://127.0.0.1:8000/health")

print("Status HTTP:", resposta.status_code)
print(resposta.json())

INFO:     127.0.0.1:51688 - "GET /health HTTP/1.1" 200 OK
Status HTTP: 200
{'status': 'ok', 'modelo_carregado': True}


## Testar a predição

In [96]:
dados = {
    "tipo": "M",
    "temperatura_ar_k": 300.0,
    "temperatura_processo_k": 310.0,
    "velocidade_rotacao_rpm": 1500.0,
    "torque_nm": 40.0,
    "desgaste_ferramenta_min": 100.0
}

resposta = requests.post(
    "http://127.0.0.1:8000/predict",
    json=dados
)

print("Status HTTP:", resposta.status_code)
print(resposta.json())

INFO:     127.0.0.1:51690 - "POST /predict HTTP/1.1" 200 OK
Status HTTP: 200
{'falha_maquina': 0, 'probabilidade_falha': 0.0005567988215874698}


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## testar a validação

In [97]:
dados_invalidos = {
    "tipo": "M",
    "temperatura_ar_k": 300.0,
    "temperatura_processo_k": 310.0,
    "velocidade_rotacao_rpm": 1500.0,
    "torque_nm": -40.0,
    "desgaste_ferramenta_min": 100.0
}

resposta = requests.post(
    "http://127.0.0.1:8000/predict",
    json=dados_invalidos
)

print("Status HTTP:", resposta.status_code)
print(resposta.json())

INFO:     127.0.0.1:51694 - "POST /predict HTTP/1.1" 422 Unprocessable Content
Status HTTP: 422
{'detail': [{'type': 'greater_than_equal', 'loc': ['body', 'torque_nm'], 'msg': 'Input should be greater than or equal to 0', 'input': -40.0, 'ctx': {'ge': 0.0}}]}


In [100]:
dados_invalidos = {
    "tipo": "X",
    "temperatura_ar_k": 300.0,
    "temperatura_processo_k": 310.0,
    "velocidade_rotacao_rpm": 1500.0,
    "torque_nm": 40.0,
    "desgaste_ferramenta_min": 100.0
}

resposta = requests.post(
    "http://127.0.0.1:8000/predict",
    json=dados_invalidos
)

print("Status HTTP:", resposta.status_code)
print(resposta.json())

INFO:     127.0.0.1:52060 - "POST /predict HTTP/1.1" 422 Unprocessable Content
Status HTTP: 422
{'detail': [{'type': 'literal_error', 'loc': ['body', 'tipo'], 'msg': "Input should be 'L', 'M' or 'H'", 'input': 'X', 'ctx': {'expected': "'L', 'M' or 'H'"}}]}


In [101]:
import os

for arquivo in sorted(os.listdir(".")):
    if os.path.isfile(arquivo):
        print(arquivo)

main.py
manutencao_preditiva.csv
modelo_falhas.joblib
routers.py
schemas.py
services.py
